# 2 · Construcción del primer agente

**Objetivo:** pasar de «llamar a un LLM» a un **agente**: un LLM que decide qué herramientas usar,
las ejecuta y vuelve a pensar con el resultado.

**Empezar aquí sin haber hecho la sección 1:** no necesitas nada de antes. Solo tu `.env` configurado
(ver `00_preparacion.ipynb`).

In [ ]:
from taller import get_llm
from taller.viz import mostrar_grafo, mostrar_mensajes

llm = get_llm()
respuesta = llm.invoke("En una frase: ¿qué es un agente de IA?")
respuesta.text

## 2.1 · Herramientas

Una herramienta es una función de Python con `@tool`. El LLM **no la ejecuta**: solo ve su nombre,
su *docstring* y sus parámetros, y **pide** que la llamemos. La docstring es parte del prompt:
escríbela pensando en el modelo.

In [ ]:
from datetime import date, timedelta

from langchain.tools import tool

from taller.tools import hora_actual  # devuelve fecha y hora en Madrid


@tool
def dias_laborables(inicio: str, fin: str) -> int:
    """Cuenta los días laborables (lunes a viernes) entre dos fechas, ambas incluidas.

    Args:
        inicio: fecha inicial en formato AAAA-MM-DD.
        fin: fecha final en formato AAAA-MM-DD.
    """
    start, end = date.fromisoformat(inicio), date.fromisoformat(fin)
    return sum((start + timedelta(d)).weekday() < 5 for d in range((end - start).days + 1))


tools = [hora_actual, dias_laborables]
llm_con_tools = llm.bind_tools(tools)

msg = llm_con_tools.invoke("¿Cuántos días laborables hay del 2 al 13 de noviembre de 2026?")
msg.tool_calls  # el LLM pide una llamada; nadie la ha ejecutado todavía

## 2.2 · Un agente «a mano» (sin LangGraph)

El bucle de un agente cabe en pocas líneas: llamar al LLM → si pide herramientas, ejecutarlas →
devolverle los resultados → repetir hasta que responda. El `print` enseña cada vuelta del bucle: qué herramienta pide el modelo y qué
recibe.

In [ ]:
from langchain.messages import HumanMessage, ToolMessage

tools_by_name = {t.name: t for t in tools}


def agente_a_mano(pregunta: str) -> str:
    messages = [HumanMessage(pregunta)]
    while True:
        ai = llm_con_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.text
        for call in ai.tool_calls:
            result = tools_by_name[call["name"]].invoke(call["args"])
            print(f"🔧 {call['name']}({call['args']}) → {result}")
            messages.append(ToolMessage(str(result), tool_call_id=call["id"]))


agente_a_mano("Si me cojo libre del 2 al 13 de noviembre de 2026, ¿cuántos días laborables gasto?")

Funciona. Pero piensa en lo que le falta para producción:

- **Memoria:** si se reinicia el proceso, la conversación se pierde.
- **Supervisión humana:** ¿cómo pausamos para que alguien apruebe una acción y seguimos *horas después*?
- **Streaming:** ¿cómo enseñamos al usuario qué está pasando paso a paso?
- **Control:** límites de iteraciones, reintentos, pasos obligatorios (recuperar antes de responder)...
- **Observabilidad:** ¿qué hizo el agente y por qué?

Cada requisito añade `if`s a ese `while`. **LangGraph convierte el bucle en un grafo explícito**
y te da todo eso de serie.

## 2.3 · El mismo agente con LangGraph

- `MessagesState`: un estado con una lista `messages` y el reducer `add_messages`, que añade mensajes
  (y actualiza los que tienen el mismo id) en lugar de sobrescribir la lista.
- `ToolNode`: un nodo ya hecho que ejecuta las herramientas que pide el último mensaje.
- `tools_condition`: enruta a `"tools"` si hay llamadas a herramientas y a `END` si no.

In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition


def agent(state: MessagesState) -> dict:
    return {"messages": [llm_con_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # → "tools" o END
builder.add_edge("tools", "agent")  # ← el ciclo
agente = builder.compile()

mostrar_grafo(agente)

In [ ]:
resultado = agente.invoke(
    {"messages": [("human", "¿Qué día es hoy? ¿Cuántos días laborables quedan hasta el 31 de diciembre?")]}
)
mostrar_mensajes(resultado["messages"])

## 2.4 · Streaming

Podemos combinar varios modos: `updates` (qué devuelve cada nodo) y `messages` (tokens del LLM según se
generan). Con varios modos, cada evento es una tupla `(modo, datos)`.

In [ ]:
pregunta = {"messages": [("human", "¿Qué hora es en Madrid?")]}
for modo, datos in agente.stream(pregunta, stream_mode=["updates", "messages"]):
    if modo == "updates":
        for nodo in datos:
            print(f"\n── nodo «{nodo}» terminado")
    elif modo == "messages":
        chunk, metadata = datos
        print(chunk.text, end="", flush=True)

## 2.5 · Verlo en directo: Studio

Hasta ahora hemos ejecutado el grafo **dentro** del notebook. El mismo grafo se puede **servir**: `langgraph.json`
(en la raíz del repositorio) le dice al *Agent Server* dónde están los grafos, y **Studio** es la interfaz para
ejecutarlos y verlos paso a paso: el dibujo del grafo, el estado después de cada nodo, cada llamada a herramienta
con sus argumentos y su resultado.

1. En **otra terminal**, desde la raíz del repositorio: `uv run studio`
2. Se abre Studio en el navegador (o entra en <https://eu.smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024>).
   Necesita una cuenta gratuita de LangSmith; con `LANGSMITH_TRACING=false` en `.env` nada sale de tu máquina.
3. Elige el grafo **agente_basico**, escribe «¿Cuántos días laborables quedan hasta fin de mes?» y pulsa *Submit*.

El servidor **recarga en caliente**: cuando hagas el ejercicio 2, el agente de Studio tendrá tu herramienta nueva
en cuanto la añadas a la lista `TOOLS` de `taller/studio.py`.

Desde Python se habla con el servidor igual que lo hace Studio, con el **SDK de LangGraph** (`langgraph_sdk`): es
la misma API HTTP que tendría el grafo desplegado.

In [ ]:
from taller.servidor import preguntar

# Si el servidor no está arrancado, esta celda te dice cómo hacerlo y no falla.
preguntar("agente_basico", "¿Qué hora es en Madrid y cuántos días laborables quedan hasta fin de mes?")

### ✍️ Ejercicio 2

Crea una herramienta `siguiente_dia_laborable(fecha: str) -> str` que devuelva el siguiente día
laborable después de una fecha. Añádela a `tools`, vuelve a enlazarla al LLM (`bind_tools`) y
reconstruye el grafo. Pregunta: *«Si hoy despliego mal, ¿cuál es el siguiente día laborable?»*

Cuando funcione en el notebook, añádela también a `TOOLS` en `taller/studio.py` y pruébala en Studio: el servidor se recarga solo.

In [ ]:
# TODO: crea la herramienta, añádela y reconstruye el agente

<details><summary><b>Ver solución</b></summary>

```python
@tool
def siguiente_dia_laborable(fecha: str) -> str:
    """Devuelve el siguiente día laborable (lunes a viernes) posterior a una fecha AAAA-MM-DD."""
    d = date.fromisoformat(fecha) + timedelta(days=1)
    while d.weekday() >= 5:
        d += timedelta(days=1)
    return d.isoformat()

tools = [hora_actual, dias_laborables, siguiente_dia_laborable]
llm_con_tools = llm.bind_tools(tools)  # `agent` usa esta variable global

builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
agente = builder.compile()
mostrar_mensajes(agente.invoke({"messages": [("human", "Si hoy despliego mal, ¿cuál es el siguiente día laborable?")]})["messages"][-1:])
```
</details>

## 2.6 · ¿Y `create_agent`?

LangChain 1.x ofrece `create_agent`: **el mismo bucle**, construido sobre LangGraph, en una línea, y
ampliable con *middleware* (resúmenes del historial, reintentos, aprobación humana, límites...).

**Regla práctica:** empieza con `create_agent` si el bucle estándar te vale. Baja a `StateGraph`
cuando necesites **tu propio flujo**: pasos obligatorios, varias fases, ramas, subagentes... que es lo
que haremos en el resto del taller.

In [ ]:
from langchain.agents import create_agent

agente_rapido = create_agent(llm, tools, system_prompt="Eres un asistente conciso.")
mostrar_grafo(agente_rapido)
agente_rapido.invoke({"messages": [("human", "¿Qué día de la semana es hoy?")]})["messages"][-1].text

> **Para recordar:** un agente es un **ciclo** LLM ⇄ herramientas. LangGraph lo hace explícito:
> el estado son los mensajes, el LLM es un nodo, las herramientas son otro nodo y una arista
> condicional cierra el bucle.

### 🚀 Reto opcional

1. Añade un `SystemMessage` con instrucciones al principio de la lista que recibe el LLM en `agent`.
2. Ejecuta el agente con `config={"recursion_limit": 3}` y una pregunta que necesite varias
   herramientas. ¿Qué error aparece? ¿Para qué sirve?